# Notebook 03: Deep Learning U-Net Training & Segmentation

**Model B** is a compact U-Net trained on random 256 px crops (zoom, flips, rotations, colour jitter, simulated turbidity) with a BCE + soft Dice loss. Train and validation Dice are measured on full images every epoch, and early stopping keeps the epoch with the lowest validation loss.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
import matplotlib.pyplot as plt
from src.utils.config import load_config
from src.utils.reproducibility import get_device
from src.data.split_dataset import load_split_manifest
from src.models.cnn import HullUNet

config = load_config()
dev = get_device('auto')
print('Using device:', dev)
splits = load_split_manifest('../data/processed/split_manifest_mavecodd.json')


## 1. Model size


In [ ]:
unet = HullUNet(base_channels=config['cnn_model']['base_channels'])
print(f"Trainable parameters: {sum(p.numel() for p in unet.parameters()):,}")


## 2. Learning curves of the trained model

Full training takes ~40 min on CPU: `python scripts/train_model.py --dataset mavecodd`. To retrain here instead, use `src.models.train.train_unet(splits['train'], splits['val'], config=config)`.


In [ ]:
summary = json.load(open('../outputs/metrics/training_summary_mavecodd.json'))
h = summary['unet']['history']
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h['epoch'], h['train_loss'], 'o-', label='train'); ax[0].plot(h['epoch'], h['val_loss'], 's-', label='val')
ax[0].set_title('Loss'); ax[0].legend()
ax[1].plot(h['epoch'], h['train_dice'], 'o-', label='train'); ax[1].plot(h['epoch'], h['val_dice'], 's-', label='val')
ax[1].set_title('Dice'); ax[1].set_ylim(0, 1); ax[1].legend()
plt.show()
cal = summary['unet']['calibration']
print('Kept epoch:', summary['unet']['best_epoch'], '| pixel threshold:', cal['mask_threshold'],
      '| area threshold:', cal['patch_area_threshold'])
